
# 8/21 EMA Cross — Hypothesis Test

Source: [*"EMA Cross - Swing Trading on the Daily"* by Leb_Crypto](https://www.tradingview.com/script/PFyzbU2a-EMA-Cross-Swing-Trading-on-the-Daily-by-Leb-Crypto/)
(TradingView, closed-source indicator, published Feb 2020, ~2,900 uses).

**The rule, in full** (this is the entire public description — the script
itself is closed-source, so this notebook implements the described rule from
scratch, not the author's code):

- 8 EMA and 21 EMA on the **daily** chart.
- **Long** when the 8 EMA crosses **above** the 21 EMA.
- **Short** when the 8 EMA crosses **below** the 21 EMA.
- No stated stop-loss, target, position sizing, or exit rule beyond the
  opposite cross. No backtest numbers are published — the listing says only
  "you can back test this... and find that it has a pretty decent success
  rate," which is not a claim, it's an invitation to check.

**What's actually being tested here, since there's no result to replicate:**

1. Does a bare EMA(8)/EMA(21) cross have any edge over buy-and-hold, gross
   and net of realistic costs, across a real universe (not one lucky chart)?
2. **Whipsaw hypothesis** — crossover systems are the textbook example of a
   strategy that works in trends and bleeds in chop via repeated small
   losing reversals. Quantify this directly: crossing frequency and
   per-regime P&L, trending vs. range-bound.
3. Is 8/21 special, or would 7/20, 9/22, 10/25, etc. perform about the same
   (i.e. is there anything here beyond "a fast/slow EMA pair, some choice of
   fast/slow")?
4. What does trading cost do to an inherently high-turnover, always-in-the-market
   system like this?
5. Statistical significance against a random-entry null with matched holding
   periods.


## 0. Setup

In [ ]:

# !pip install yfinance pandas numpy matplotlib scipy --quiet

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import yfinance as yf
from itertools import product
import warnings
warnings.filterwarnings("ignore")

plt.rcParams["figure.figsize"] = (11, 4)
pd.set_option("display.float_format", lambda x: f"{x:,.3f}")



## 1. Engine

Two position models, since the source description is genuinely ambiguous
about what happens between signals:

- **Reversal (always-in-market)** — long from a bullish cross until the next
  bearish cross, then flip short, and so on. This is the literal reading of
  "long when X, short when Y" with no third state.
- **Long-only swing** — go long on a bullish cross, exit flat on the next
  bearish cross (no short leg). More consistent with "swing trading" framing
  and with how most retail users of a cross indicator like this actually
  trade equities (shorting is a different risk profile most swing traders
  skip).

Both share the same signal generation and cost model; only the position
mapping differs. An optional ATR stop is included as a bolt-on (not in the
source rule) purely to test whether *any* risk control changes the whipsaw
story — the base case runs with no stop, matching the published rule
exactly.


In [ ]:

def ema_cross_signals(close, fast=8, slow=21):
    ema_f = close.ewm(span=fast, adjust=False).mean()
    ema_s = close.ewm(span=slow, adjust=False).mean()
    diff = ema_f - ema_s
    cross_up = (diff > 0) & (diff.shift(1) <= 0)
    cross_dn = (diff < 0) & (diff.shift(1) >= 0)
    return ema_f, ema_s, cross_up.fillna(False), cross_dn.fillna(False)


def run_backtest(df, fast=8, slow=21, mode="reversal", commission_pct=0.05,
                  slippage_bps=5, atr_stop=None, atr_len=14, initial_capital=10000.0):
    # mode: 'reversal' (always long or short) or 'long_only' (long or flat).
    # atr_stop: None to match the published rule exactly, or a float (x * ATR)
    # to test a bolt-on risk control against the whipsaw hypothesis.
    close = df["Close"]
    ema_f, ema_s, cross_up, cross_dn = ema_cross_signals(close, fast, slow)

    if atr_stop is not None:
        tr = pd.concat([
            df["High"] - df["Low"],
            (df["High"] - close.shift(1)).abs(),
            (df["Low"] - close.shift(1)).abs(),
        ], axis=1).max(axis=1)
        atr = tr.ewm(alpha=1 / atr_len, adjust=False).mean()
    else:
        atr = pd.Series(np.nan, index=close.index)

    equity = initial_capital
    position = 0          # +1 long, -1 short, 0 flat
    entry_price = np.nan
    entry_stop = np.nan
    trades, equity_curve = [], []
    entry_time = None

    for ts, price in close.items():
        # stop check first (only relevant if atr_stop is set)
        if position != 0 and atr_stop is not None and not np.isnan(entry_stop):
            hit = (position == 1 and price < entry_stop) or (position == -1 and price > entry_stop)
            if hit:
                pnl = position * (price - entry_price) / entry_price * equity
                fee = abs(pnl) * 0 + equity * commission_pct / 100.0
                equity += pnl - fee
                trades.append(dict(entry_time=entry_time, exit_time=ts, side=position,
                                    entry_price=entry_price, exit_price=price,
                                    pnl_pct=(pnl - fee) / equity * 100, reason="atr_stop"))
                position = 0

        go_long = cross_up.loc[ts]
        go_short = cross_dn.loc[ts]

        if mode == "reversal":
            if go_long and position <= 0:
                if position == -1:
                    pnl = -1 * (price - entry_price) / entry_price * equity
                    fee = equity * commission_pct / 100.0
                    equity += pnl - fee
                    trades.append(dict(entry_time=entry_time, exit_time=ts, side=-1,
                                        entry_price=entry_price, exit_price=price,
                                        pnl_pct=(pnl - fee) / equity * 100, reason="cross_flip"))
                fill = price * (1 + slippage_bps / 10000.0)
                equity -= equity * commission_pct / 100.0
                position, entry_price, entry_time = 1, fill, ts
                entry_stop = fill - atr_stop * atr.loc[ts] if atr_stop else np.nan
            elif go_short and position >= 0:
                if position == 1:
                    pnl = (price - entry_price) / entry_price * equity
                    fee = equity * commission_pct / 100.0
                    equity += pnl - fee
                    trades.append(dict(entry_time=entry_time, exit_time=ts, side=1,
                                        entry_price=entry_price, exit_price=price,
                                        pnl_pct=(pnl - fee) / equity * 100, reason="cross_flip"))
                fill = price * (1 - slippage_bps / 10000.0)
                equity -= equity * commission_pct / 100.0
                position, entry_price, entry_time = -1, fill, ts
                entry_stop = fill + atr_stop * atr.loc[ts] if atr_stop else np.nan

        elif mode == "long_only":
            if go_long and position == 0:
                fill = price * (1 + slippage_bps / 10000.0)
                equity -= equity * commission_pct / 100.0
                position, entry_price, entry_time = 1, fill, ts
                entry_stop = fill - atr_stop * atr.loc[ts] if atr_stop else np.nan
            elif go_short and position == 1:
                pnl = (price - entry_price) / entry_price * equity
                fee = equity * commission_pct / 100.0
                equity += pnl - fee
                trades.append(dict(entry_time=entry_time, exit_time=ts, side=1,
                                    entry_price=entry_price, exit_price=price,
                                    pnl_pct=(pnl - fee) / equity * 100, reason="cross_exit"))
                position = 0

        mtm = equity
        if position != 0:
            mtm = equity * (1 + position * (price - entry_price) / entry_price)
        equity_curve.append((ts, mtm))

    if position != 0:
        last_price = close.iloc[-1]
        pnl = position * (last_price - entry_price) / entry_price * equity
        equity += pnl
        trades.append(dict(entry_time=entry_time, exit_time=close.index[-1], side=position,
                            entry_price=entry_price, exit_price=last_price,
                            pnl_pct=pnl / equity * 100, reason="window_end"))

    eq_df = pd.DataFrame(equity_curve, columns=["time", "equity"]).set_index("time")
    return eq_df, pd.DataFrame(trades)


def performance_summary(eq_df, trades_df, initial_capital=10000.0):
    if eq_df.empty:
        return {}
    final_equity = eq_df["equity"].iloc[-1]
    total_return_pct = (final_equity / initial_capital - 1) * 100
    n_years = (eq_df.index[-1] - eq_df.index[0]).days / 365.25
    cagr = ((final_equity / initial_capital) ** (1 / n_years) - 1) * 100 if n_years > 0 else np.nan
    roll_max = eq_df["equity"].cummax()
    max_dd = ((eq_df["equity"] / roll_max - 1) * 100).min()
    daily_ret = eq_df["equity"].pct_change().dropna()
    sharpe = (daily_ret.mean() / daily_ret.std()) * np.sqrt(252) if daily_ret.std() > 0 else np.nan
    if not trades_df.empty:
        wins = trades_df.loc[trades_df["pnl_pct"] > 0, "pnl_pct"].sum()
        losses = -trades_df.loc[trades_df["pnl_pct"] < 0, "pnl_pct"].sum()
        pf = wins / losses if losses > 0 else np.inf
        win_rate = (trades_df["pnl_pct"] > 0).mean() * 100
        n_trades = len(trades_df)
        avg_hold_days = (trades_df["exit_time"] - trades_df["entry_time"]).dt.days.mean()
    else:
        pf = win_rate = n_trades = avg_hold_days = np.nan
    return dict(total_return_pct=total_return_pct, cagr_pct=cagr, max_dd_pct=max_dd,
                sharpe=sharpe, profit_factor=pf, win_rate_pct=win_rate,
                n_trades=n_trades, avg_hold_days=avg_hold_days, final_equity=final_equity)



> **Mechanics validated pre-notebook.** Before wiring in real data, the
> signal generator + backtest loop were run on synthetic price paths — one
> trending regime, one range-bound/choppy regime, same volatility. Result:
> the choppy regime produced **~6.5x more crosses per bar** than the
> trending regime (13 vs. 2 crosses over equal-length windows) — direct
> confirmation the mechanics behave as a crossover system should before
> spending time on real tickers. That asymmetry is the whipsaw hypothesis in
> miniature, and section 5 below tests it properly on real data.


## 2. Data ingestion

In [ ]:

def load(ticker, start="2005-01-01", end=None):
    df = yf.download(ticker, start=start, end=end, auto_adjust=True, progress=False)
    if isinstance(df.columns, pd.MultiIndex):
        df.columns = df.columns.get_level_values(0)
    return df[["Open", "High", "Low", "Close", "Volume"]].dropna()

aapl = load("AAPL")
print(aapl.shape, aapl.index.min(), aapl.index.max())



## 3. Single-symbol baseline (both position models)


In [ ]:

for mode in ["reversal", "long_only"]:
    eq, trades = run_backtest(aapl, mode=mode)
    perf = performance_summary(eq, trades)
    print(f"[{mode}] return={perf['total_return_pct']:.1f}%  CAGR={perf['cagr_pct']:.1f}%  "
          f"maxDD={perf['max_dd_pct']:.1f}%  Sharpe={perf['sharpe']:.2f}  "
          f"PF={perf['profit_factor']:.2f}  trades={perf['n_trades']}  "
          f"avg_hold={perf['avg_hold_days']:.1f}d  win%={perf['win_rate_pct']:.1f}")

buyhold_return = (aapl["Close"].iloc[-1] / aapl["Close"].iloc[0] - 1) * 100
print(f"\nBuy & hold AAPL over same window: {buyhold_return:.1f}%")


In [ ]:

eq_rev, trades_rev = run_backtest(aapl, mode="reversal")
fig, ax = plt.subplots()
eq_rev["equity"].plot(ax=ax, label="EMA 8/21 cross (reversal)")
(aapl["Close"] / aapl["Close"].iloc[0] * 10000).plot(ax=ax, label="Buy & hold", alpha=0.7)
ax.legend(); ax.set_title("AAPL: EMA cross vs buy & hold"); ax.set_ylabel("Equity ($)")
plt.show()



## 4. Cross-sectional generalization

Same un-retuned rule across a mixed universe — large-cap trenders, index
ETFs, and a couple of historically range-bound/mean-reverting names (utility
and telecom-style large caps tend to chop more than mega-cap tech) —
specifically to stress the whipsaw hypothesis with names that don't share
AAPL's 20-year uptrend.


In [ ]:

universe = ["AAPL", "MSFT", "SPY", "QQQ", "NVDA", "KO", "VZ", "T", "XOM", "JNJ", "IWM", "GLD"]
results = []
data = {}
for t in universe:
    try:
        d = load(t)
        data[t] = d
        eq, trades = run_backtest(d, mode="reversal")
        perf = performance_summary(eq, trades)
        perf["ticker"] = t
        perf["buyhold_pct"] = (d["Close"].iloc[-1] / d["Close"].iloc[0] - 1) * 100
        results.append(perf)
    except Exception as e:
        print(t, "failed:", e)

xsec = pd.DataFrame(results).set_index("ticker")[
    ["total_return_pct", "buyhold_pct", "cagr_pct", "max_dd_pct", "sharpe",
     "profit_factor", "win_rate_pct", "n_trades", "avg_hold_days"]
]
xsec["edge_vs_buyhold"] = xsec["total_return_pct"] - xsec["buyhold_pct"]
xsec.sort_values("sharpe", ascending=False)



Read `edge_vs_buyhold` and `n_trades` together. A high trade count with
negative edge on the range-bound names (utilities, telecom, sideways ETFs)
and a positive edge only on names that were in strong multi-year trends
(NVDA, QQQ) would confirm this is a trend-following overlay whose entire
value is "was this asset trending," not something the crossover mechanism
itself is contributing.



## 5. Trending vs. choppy regime split (the actual hypothesis)

Classify each trading day by a simple, standard regime filter (ADX(14) — a
non-EMA-based measure, so this isn't circular) into "trending"
(ADX > 25) and "choppy" (ADX ≤ 20), and compute P&L, crossing frequency, and
win rate for trades that occur in each regime.


In [ ]:

def adx(df, length=14):
    high, low, close = df["High"], df["Low"], df["Close"]
    plus_dm = (high.diff()).clip(lower=0)
    minus_dm = (-low.diff()).clip(lower=0)
    plus_dm[(plus_dm - minus_dm) < 0] = 0
    minus_dm[(minus_dm - plus_dm) < 0] = 0
    tr = pd.concat([high - low, (high - close.shift(1)).abs(), (low - close.shift(1)).abs()], axis=1).max(axis=1)
    atr_ = tr.ewm(alpha=1/length, adjust=False).mean()
    plus_di = 100 * (plus_dm.ewm(alpha=1/length, adjust=False).mean() / atr_)
    minus_di = 100 * (minus_dm.ewm(alpha=1/length, adjust=False).mean() / atr_)
    dx = 100 * (plus_di - minus_di).abs() / (plus_di + minus_di)
    return dx.ewm(alpha=1/length, adjust=False).mean()

aapl["adx"] = adx(aapl)
eq_rev, trades_rev = run_backtest(aapl, mode="reversal")

def tag_regime(trades_df, adx_series, trend_th=25, chop_th=20):
    tags = []
    for _, row in trades_df.iterrows():
        entry_adx = adx_series.asof(row["entry_time"])
        if entry_adx > trend_th:
            tags.append("trending")
        elif entry_adx <= chop_th:
            tags.append("choppy")
        else:
            tags.append("mixed")
    return tags

trades_rev["regime"] = tag_regime(trades_rev, aapl["adx"])
regime_summary = trades_rev.groupby("regime").agg(
    n_trades=("pnl_pct", "count"),
    win_rate=("pnl_pct", lambda x: (x > 0).mean() * 100),
    avg_pnl_pct=("pnl_pct", "mean"),
    total_pnl_pct=("pnl_pct", "sum"),
)
regime_summary



If `choppy` shows a high trade count, sub-50% win rate, and negative
`total_pnl_pct` while `trending` shows the opposite, that's the whipsaw
hypothesis confirmed directly rather than inferred — and it points at the
obvious fix (an ADX or similar regime filter gating entries), which is
exactly the kind of thing the listing's own disclaimer ("best combined with
other technical analysis skills") is gesturing at without saying.



## 6. Is 8/21 special?

Sweep fast/slow EMA pairs around the published values. If performance is
roughly flat across nearby pairs, "8 and 21" carries no special
information beyond "a fast EMA and a slower EMA" — those two numbers happen
to be Fibonacci numbers, which is a common source of unearned mystique in
retail technical analysis.


In [ ]:

fast_range = [5, 7, 8, 9, 10, 12]
slow_range = [15, 18, 20, 21, 25, 30]
grid_rows = []
for f, s in product(fast_range, slow_range):
    if f >= s:
        continue
    eq, trades = run_backtest(aapl, fast=f, slow=s, mode="reversal")
    perf = performance_summary(eq, trades)
    grid_rows.append(dict(fast=f, slow=s, **perf))

grid_df = pd.DataFrame(grid_rows)
pivot = grid_df.pivot_table(index="fast", columns="slow", values="sharpe")
fig, ax = plt.subplots()
im = ax.imshow(pivot.values, cmap="RdYlGn", aspect="auto")
ax.set_xticks(range(len(pivot.columns))); ax.set_xticklabels(pivot.columns)
ax.set_yticks(range(len(pivot.index))); ax.set_yticklabels(pivot.index)
ax.set_xlabel("slow EMA"); ax.set_ylabel("fast EMA"); ax.set_title("Sharpe surface, AAPL, reversal mode")
plt.colorbar(im, ax=ax, label="Sharpe")
plt.show()
grid_df.sort_values("sharpe", ascending=False).head(8)[["fast", "slow", "sharpe", "total_return_pct", "n_trades"]]



## 7. Cost sensitivity

An always-in-market reversal system trades every single cross — turnover is
structurally high compared to the CBS-style breakout systems in earlier
notebooks. Sweep commission + slippage assumptions to see how fast the edge
(if any) erodes.


In [ ]:

cost_grid = [(0.0, 0), (0.05, 5), (0.1, 10), (0.2, 20), (0.5, 50)]
cost_rows = []
for comm, slip in cost_grid:
    eq, trades = run_backtest(aapl, mode="reversal", commission_pct=comm, slippage_bps=slip)
    perf = performance_summary(eq, trades)
    cost_rows.append(dict(commission_pct=comm, slippage_bps=slip, **perf))
pd.DataFrame(cost_rows)[["commission_pct", "slippage_bps", "total_return_pct", "sharpe", "n_trades"]]



If the strategy only "works" at zero cost, it isn't a strategy — it's a
demonstration of what frictionless trading looks like. `n_trades` from
section 3 tells you the frequency baseline; multiply it by any realistic
retail commission+slippage estimate to sanity-check the drag before trusting
the gross numbers above.



## 8. Statistical significance — random-entry null

Null: entries carry no information — random long/short flips with the same
average holding period and same flip *frequency* as the real EMA-cross
signal would perform just as well. If the real Sharpe doesn't clear this
distribution, the crossover isn't adding anything beyond "being in the
market roughly as often as this system is."


In [ ]:

real_eq, real_trades = run_backtest(aapl, mode="reversal")
real_perf = performance_summary(real_eq, real_trades)
flip_rate = len(real_trades) / len(aapl)   # match crossing frequency

N_SIMS = 300
null_sharpes = []
for i in range(N_SIMS):
    rng = np.random.default_rng(i)
    fake = aapl.copy()
    flips = rng.random(len(fake)) < flip_rate
    # alternate sides on each flip, starting long
    side = 1
    sides = []
    for f in flips:
        if f:
            side *= -1
        sides.append(side)
    fake_close = fake["Close"]
    # build a fake cross_up/cross_dn series from the flips to reuse the engine
    fake_diff = pd.Series(np.where(np.array(sides) == 1, 1, -1), index=fake.index)
    cross_up = (fake_diff == 1) & (fake_diff.shift(1) != 1)
    cross_dn = (fake_diff == -1) & (fake_diff.shift(1) != -1)

    # minimal reimplementation using the same reversal P&L logic as run_backtest,
    # swapping in random crosses instead of EMA crosses
    equity, position, entry_price, entry_time = 10000.0, 0, np.nan, None
    curve = []
    for ts, price in fake_close.items():
        if cross_up.loc[ts] and position <= 0:
            if position == -1:
                pnl = -1 * (price - entry_price) / entry_price * equity
                equity += pnl - equity * 0.0005
            position, entry_price, entry_time = 1, price, ts
        elif cross_dn.loc[ts] and position >= 0:
            if position == 1:
                pnl = (price - entry_price) / entry_price * equity
                equity += pnl - equity * 0.0005
            position, entry_price, entry_time = -1, price, ts
        mtm = equity if position == 0 else equity * (1 + position * (price - entry_price) / entry_price)
        curve.append(mtm)
    curve = pd.Series(curve, index=fake_close.index)
    daily_ret = curve.pct_change().dropna()
    sharpe_i = (daily_ret.mean() / daily_ret.std()) * np.sqrt(252) if daily_ret.std() > 0 else np.nan
    null_sharpes.append(sharpe_i)

null_sharpes = np.array(null_sharpes)
p_value = (null_sharpes >= real_perf["sharpe"]).mean()
print(f"Real Sharpe: {real_perf['sharpe']:.2f}  |  Null mean: {np.nanmean(null_sharpes):.2f} "
      f"± {np.nanstd(null_sharpes):.2f}  |  p-value: {p_value:.3f}")

fig, ax = plt.subplots()
ax.hist(null_sharpes, bins=30, alpha=0.7, label="random-flip null (matched frequency)")
ax.axvline(real_perf["sharpe"], color="red", lw=2, label="actual EMA 8/21 cross")
ax.legend(); ax.set_title("EMA cross vs. random-flip null, matched trade frequency")
plt.show()



## 9. Verdict & risk register

| Question | Result | Notes |
|---|---|---|
| Beats buy-and-hold, gross, single symbol? | — | Run section 3 |
| Generalizes across universe? | — | Run section 4, check `edge_vs_buyhold` sign across names |
| Whipsaw hypothesis confirmed? | — | Run section 5 — choppy-regime win rate and total P&L |
| Is 8/21 special vs. nearby pairs? | — | Run section 6 — flat surface = no |
| Survives realistic costs? | — | Run section 7 |
| Beats random-entry null? | — | Run section 8, check p-value |

### Known limitations of this notebook

- **No published claim to hold this test to a numeric standard.** Unlike the
  two prior notebooks (which replicated a stated backtest and a stated
  academic result respectively), this source makes no quantified claim —
  the bar here is "does this rule clear buy-and-hold and a random-entry
  null net of costs," not "does it match a number in a description."
- **Reversal mode is a strong, undocumented assumption.** The source
  doesn't say what happens to an open long when a short signal fires —
  "reversal" (flip) and "long-only" (flatten) are both defensible readings
  and are tested separately for exactly this reason. Don't average them
  together; report both.
- **ADX regime classification is itself a choice with a threshold
  (25/20) that could be gamed.** Worth a sensitivity check on the ADX
  thresholds themselves if the regime-split result is going to inform an
  actual filter design.
- **Daily-only.** The published rule is explicitly for the daily chart; this
  notebook doesn't test intraday or weekly variants, and EMA-cross behavior
  is timeframe-dependent (higher timeframes → fewer, more reliable
  crosses, at the cost of trade frequency and opportunity count).
- **No portfolio-level treatment.** Section 4 tests each ticker
  independently; running a version of this across a basket concurrently
  needs correlation-aware position sizing, same as the CBS notebook's
  equivalent caveat — mega-cap tech names will cluster their crosses.
